# Unit 6: BrainFlow ML 集成

## 学习目标
- 理解 BrainFlow 的 `MLModel` 模块
- 掌握 `BrainFlowMetrics`（放松度、专注度、正念度）
- 使用 `DataFilter.get_avg_band_powers()` 作为 ML 输入
- 了解 ONNX 自定义模型集成
- 在滑动窗口上持续推理，模拟实时评估

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from brainflow.board_shim import BoardShim, BrainFlowInputParams, BoardIds
from brainflow.data_filter import (
    DataFilter, FilterTypes, DetrendOperations, WindowOperations
)
from brainflow.ml_model import (
    MLModel, BrainFlowMetrics, BrainFlowClassifiers, BrainFlowModelParams
)

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print("模块导入完成")

## 6.1 BrainFlow ML 模型概览

BrainFlow 内置了预训练的 ML 模型，可以直接用于心理状态评估：

| BrainFlowMetrics | 含义 | 输出 |
|-------------------|------|------|
| `RESTFULNESS` (0) | **放松度** | 0（紧张）~ 1（放松） |
| `MENTAL_FOCUS` (1) | **专注度** (Mental Effort) | 0（不专注）~ 1（高度专注） |
| `MINDFULNESS` (2) | **正念度** | 0 ~ 1 |
| `USER_DEFINED` (3) | 用户自定义模型 | 取决于模型 |

| BrainFlowClassifiers | 说明 |
|----------------------|------|
| `REGRESSION` (0) | 回归（输出连续值） |
| `KNN` (1) | K 近邻 |
| `SVM` (2) | 支持向量机 |
| `LDA` (3) | 线性判别分析 |
| `ONNX` (-1) | 自定义 ONNX 模型 |

## 6.2 准备 EEG 数据

ML 模型需要频带功率特征作为输入。

In [ ]:
# 采集数据
board = BoardShim(BoardIds.SYNTHETIC_BOARD, BrainFlowInputParams())
board.prepare_session()
board.start_stream()
time.sleep(15)
data = board.get_board_data()
board.stop_stream()
board.release_session()

descr = BoardShim.get_board_descr(BoardIds.SYNTHETIC_BOARD)
eeg_channels = descr['eeg_channels']
eeg_names = BoardShim.get_eeg_names(BoardIds.SYNTHETIC_BOARD)
sampling_rate = descr['sampling_rate']

# 提取 EEG 数据并预处理
def preprocess_channel(data, ch_idx, sr):
    signal = data[ch_idx].copy().astype(np.float64)
    DataFilter.detrend(signal, DetrendOperations.LINEAR.value)
    DataFilter.perform_bandpass(signal, sr, 22.5, 45.0, 4,
                                 FilterTypes.BUTTERWORTH.value, 0)
    DataFilter.perform_bandstop(signal, sr, 50.0, 4.0, 4,
                                 FilterTypes.BUTTERWORTH.value, 0)
    return signal

eeg_data = np.array([preprocess_channel(data, ch, sampling_rate)
                      for ch in eeg_channels])

print(f"数据形状: {eeg_data.shape}")
print(f"采样率: {sampling_rate} Hz")
print(f"时长: {eeg_data.shape[1] / sampling_rate:.1f} 秒")

## 6.3 计算频带功率特征

BrainFlow ML 模型的标准输入是 5 个频带（Delta, Theta, Alpha, Beta, Gamma）的平均功率。
\使用 `get_avg_band_powers()` 一次性计算所有通道的所有频带。

In [ ]:
# 使用 get_avg_band_powers 计算频带功率
# 参数说明:
#   data: 2D 数组 (n_channels, n_samples)
#   channels: 本地通道索引列表 [0, 1, 2, ...]
#   sampling_rate: 采样率
#   apply_filters: True 表示内部自动应用滤波器

result = DataFilter.get_avg_band_powers(
    eeg_data,
    list(range(len(eeg_channels))),
    sampling_rate,
    True
)

# result[0]: 5个频带的平均功率 [delta, theta, alpha, beta, gamma]
avg_band_powers = result[0]
stddev_band_powers = result[1]

print(f"频带功率向量形状: {avg_band_powers.shape}")
print(f"\n频带功率 (所有通道平均):")
band_names = ['Delta (0.5-4Hz)', 'Theta (4-8Hz)', 'Alpha (8-13Hz)', 
              'Beta (13-30Hz)', 'Gamma (30-50Hz)']
for i, name in enumerate(band_names):
    print(f"  {name:<20}: {avg_band_powers[i]:.4f} ± {stddev_band_powers[i]:.4f}")

## 6.4 评估放松度 (Restfulness)

使用预训练的放松度模型评估当前心理状态。

In [ ]:
# 创建放松度评估模型
print("初始化放松度模型...")
restfulness_params = BrainFlowModelParams(
    BrainFlowMetrics.RESTFULNESS.value,
    BrainFlowClassifiers.REGRESSION.value
)

restfulness_model = MLModel(restfulness_params)
restfulness_model.prepare()
print("✅ 放松度模型就绪")

# 预测
restfulness_score = restfulness_model.predict(avg_band_powers)
# predict 在新版本返回数组，取第一个元素
if isinstance(restfulness_score, (list, np.ndarray)):
    restfulness_score = restfulness_score[0]

restfulness_model.release()

print(f"\n📊 放松度分数: {restfulness_score:.4f}")
print(f"   范围: 0.0 (紧张) ~ 1.0 (放松)")
print(f"   解读: {'😌 放松' if restfulness_score > 0.5 else '😰 紧张'}")

## 6.5 评估专注度 (Mental Focus)

专注度模型评估认知投入水平。

In [ ]:
# 创建专注度评估模型
print("初始化专注度模型...")
focus_params = BrainFlowModelParams(
    BrainFlowMetrics.MENTAL_FOCUS.value,
    BrainFlowClassifiers.REGRESSION.value
)

focus_model = MLModel(focus_params)
focus_model.prepare()
print("✅ 专注度模型就绪")

# 预测
focus_score = focus_model.predict(avg_band_powers)
if isinstance(focus_score, (list, np.ndarray)):
    focus_score = focus_score[0]

focus_model.release()

print(f"\n📊 专注度分数: {focus_score:.4f}")
print(f"   范围: 0.0 (不专注) ~ 1.0 (高度专注)")
print(f"   解读: {'🧠 专注' if focus_score > 0.5 else '💤 不专注'}")

## 6.6 评估正念度 (Mindfulness)

正念度模型评估冥想/正念状态。

In [ ]:
# 创建正念度评估模型
print("初始化正念度模型...")
mindfulness_params = BrainFlowModelParams(
    BrainFlowMetrics.MINDFULNESS.value,
    BrainFlowClassifiers.REGRESSION.value
)

mindfulness_model = MLModel(mindfulness_params)
mindfulness_model.prepare()
print("✅ 正念度模型就绪")

# 预测
mindfulness_score = mindfulness_model.predict(avg_band_powers)
if isinstance(mindfulness_score, (list, np.ndarray)):
    mindfulness_score = mindfulness_score[0]

mindfulness_model.release()

print(f"\n📊 正念度分数: {mindfulness_score:.4f}")
print(f"   范围: 0.0 ~ 1.0")
print(f"   解读: {'🧘 正念' if mindfulness_score > 0.5 else '🌪️ 散乱'}")

## 6.7 综合心理状态报告

将三种指标汇总成一个综合报告。

In [ ]:
# 综合评估函数
def comprehensive_assessment(eeg_data_2d, sampling_rate, eeg_channels):
    """使用三种 BrainFlow ML 模型对 EEG 数据做综合评估"""
    
    # Step 1: 计算频带功率
    result = DataFilter.get_avg_band_powers(
        eeg_data_2d,
        list(range(len(eeg_channels))),
        sampling_rate, True
    )
    band_powers = result[0]
    
    scores = {}
    
    # Step 2: 放松度
    m = MLModel(BrainFlowModelParams(
        BrainFlowMetrics.RESTFULNESS.value,
        BrainFlowClassifiers.REGRESSION.value
    ))
    m.prepare()
    s = m.predict(band_powers)
    scores['restfulness'] = float(s[0]) if isinstance(s, (list, np.ndarray)) else float(s)
    m.release()
    
    # Step 3: 专注度
    m = MLModel(BrainFlowModelParams(
        BrainFlowMetrics.MENTAL_FOCUS.value,
        BrainFlowClassifiers.REGRESSION.value
    ))
    m.prepare()
    s = m.predict(band_powers)
    scores['focus'] = float(s[0]) if isinstance(s, (list, np.ndarray)) else float(s)
    m.release()
    
    # Step 4: 正念度
    m = MLModel(BrainFlowModelParams(
        BrainFlowMetrics.MINDFULNESS.value,
        BrainFlowClassifiers.REGRESSION.value
    ))
    m.prepare()
    s = m.predict(band_powers)
    scores['mindfulness'] = float(s[0]) if isinstance(s, (list, np.ndarray)) else float(s)
    m.release()
    
    return scores

# 执行综合评估
scores = comprehensive_assessment(eeg_data, sampling_rate, eeg_channels)

# 绘制雷达图
categories = list(scores.keys())
values = list(scores.values())

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
values += values[:1]
angles += angles[:1]

ax.fill(angles, values, alpha=0.25, color='steelblue')
ax.plot(angles, values, 'o-', linewidth=2, color='steelblue')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(['放松度', '专注度', '正念度'])
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_title('综合心理状态评估', fontsize=14, pad=20)

# 标注分数
for i, (cat, val) in enumerate(zip(categories, values[:-1])):
    print(f"  {cat}: {val:.4f}")

plt.tight_layout()
plt.show()

## 6.8 滑动窗口实时推理模拟

在真实 BCI 应用中，需要随时间推移持续推理。这里模拟滑动窗口推理过程。

In [ ]:
def realtime_assessment_simulation(eeg_data_2d, sampling_rate, 
                                     window_sec=4, step_sec=1):
    """
    模拟实时心理状态评估。
    在滑动窗口上持续推理，类似于真实的实时 BCI 系统。
    """
    window_samples = int(window_sec * sampling_rate)
    step_samples = int(step_sec * sampling_rate)
    n_total = eeg_data_2d.shape[1]
    
    records = []
    
    for start in range(0, n_total - window_samples, step_samples):
        end = start + window_samples
        window = eeg_data_2d[:, start:end]
        center_time = (start + window_samples // 2) / sampling_rate
        
        # 计算频带功率
        result = DataFilter.get_avg_band_powers(
            window, 
            list(range(eeg_data_2d.shape[0])),
            sampling_rate, True
        )
        band_powers = result[0]
        
        # 推理（为简化，仅使用放松度）
        m = MLModel(BrainFlowModelParams(
            BrainFlowMetrics.RESTFULNESS.value,
            BrainFlowClassifiers.REGRESSION.value
        ))
        m.prepare()
        s = m.predict(band_powers)
        m.release()
        score = float(s[0]) if isinstance(s, (list, np.ndarray)) else float(s)
        
        records.append({
            'time': center_time,
            'restfulness': score
        })
    
    return records

# 运行模拟
records = realtime_assessment_simulation(eeg_data, sampling_rate)

# 可视化时间演变
fig, ax = plt.subplots(figsize=(14, 4))
times = [r['time'] for r in records]
scores = [r['restfulness'] for r in records]

ax.plot(times, scores, 'o-', linewidth=1.5, markersize=4, color='steelblue')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='阈值 0.5')
ax.fill_between(times, 0.5, scores, where=np.array(scores) > 0.5,
                 color='green', alpha=0.15, label='放松')
ax.fill_between(times, 0.0, scores, where=np.array(scores) <= 0.5,
                 color='red', alpha=0.15, label='紧张')

ax.set_xlabel('时间 (秒)')
ax.set_ylabel('放松度分数')
ax.set_title('滑动窗口 — 放松度时间演变 (窗口=4秒, 步长=1秒)')
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"分析了 {len(records)} 个时间窗口")
print(f"平均放松度: {np.mean(scores):.4f}")
print(f"最大放松度: {np.max(scores):.4f}")
print(f"最小放松度: {np.min(scores):.4f}")

## 6.9 完整的多指标时间演变

同时追踪三种指标的时间变化。

In [ ]:
# 多指标滑动窗口推理
window_sec = 4
step_sec = 1
window_samples = int(window_sec * sampling_rate)
step_samples = int(step_sec * sampling_rate)

timeline = []
for start in range(0, eeg_data.shape[1] - window_samples, step_samples):
    end = start + window_samples
    window = eeg_data[:, start:end]
    center_time = (start + window_samples // 2) / sampling_rate
    
    scores = comprehensive_assessment(window, sampling_rate, eeg_channels)
    scores['time'] = center_time
    timeline.append(scores)

# 绘制多指标时间曲线
fig, ax = plt.subplots(figsize=(14, 5))
times = [t['time'] for t in timeline]

ax.plot(times, [t['restfulness'] for t in timeline], 
        'o-', linewidth=1.5, markersize=4, label='放松度', color='green')
ax.plot(times, [t['focus'] for t in timeline], 
        's-', linewidth=1.5, markersize=4, label='专注度', color='blue')
ax.plot(times, [t['mindfulness'] for t in timeline], 
        '^-', linewidth=1.5, markersize=4, label='正念度', color='purple')

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('时间 (秒)')
ax.set_ylabel('分数')
ax.set_title('多指标心理状态时间演变')
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"分析了 {len(timeline)} 个时间窗口")

## 6.10 注意事项

### ML 模型使用要点

1. **输入特征**：`get_avg_band_powers()` 返回的 5 维频带功率是标准输入
2. **模型生命周期**：每个 `MLModel` 实例都需要 `prepare()` → `predict()` → `release()`
3. **predict 返回值**：新版本返回 **数组**（不是单值），使用 `result[0]` 获取
4. **合成板上的结果**：使用 Synthetic Board 时 ML 输出是**演示性质**的，不代表真实信号
5. **真实硬件**：连接到真实 EEG 设备后，ML 模型才能给出有意义的心理状态评估

### ONNX 自定义模型

BrainFlow 支持加载 ONNX 格式的自定义模型（`BrainFlowClassifiers.ONNX`），配合 `BrainFlowMetrics.USER_DEFINED` 使用。
这允许你：
- 使用 scikit-learn + skl2onnx 训练自己的模型
- 导出为 .onnx 文件
- 在 BrainFlow 中加载推理

详见：[BrainFlow ONNX 博客](https://brainflow.org/2022-06-09-onnx/)

## 小结

| 知识点 | 要点 |
|--------|------|
| `MLModel` | BrainFlow 内置 ML 推理引擎 |
| `BrainFlowMetrics` | 放松度、专注度、正念度 |
| 输入 | `get_avg_band_powers()` 的 5 维频带功率 |
| 输出 | 回归分数 0~1，越高越显著 |
| 生命周期 | prepare → predict → release |
| 实时推理 | 在滑动窗口上持续更新评估 |

→ [Unit 7: 实战项目](unit7_capstone_project.ipynb) — 完整 BCI 流水线实战